# Minimal Skimming Processor
A coffea processor that applies a skim selection using the intccms skimming infrastructure and writes filtered events to disk. Supports both `IterativeExecutor` (local) and `DaskExecutor` (distributed via AF).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    import omegaconf
except Exception:
    ! pip install omegaconf

In [ ]:
INPUT_FILE = "root://xcache//store/mc/RunIISummer20UL18NanoAODv9/TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/120000/0E9EA19A-AE0E-3149-88C3-D733240FF5AB.root"
DATASET_NAME = "ttbar"
TREE_NAME = "Events"
OUTPUT_DIR = "root://xrootd-local.unl.edu:1094//store/user/maly/"
CHUNK_SIZE = 100_000

# AF options: [iterative, coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
AF = "coffeacasa-condor"
NUM_WORKERS = 10
AUTO_CLOSE_CLIENT = False

BRANCHES = {
    "event": ["run", "luminosityBlock", "event"],
    "Muon": ["pt", "eta", "phi", "mass"],
    "PuppiMET": ["pt", "phi"],
}

In [ ]:
import hashlib
import time

import awkward as ak
import cloudpickle
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents import NanoAODSchema
from coffea.processor import ProcessorABC, Runner, IterativeExecutor, DaskExecutor

import intccms
from intccms.schema import SkimmingConfig
from intccms.skimming.io.writers import RootWriter
from intccms.skimming.pipeline.stages import build_column_list, extract_columns, save_events
from intccms.utils.dask_client import acquire_client, live_prints
from intccms.utils.functors import SelectionExecutor

cloudpickle.register_pickle_by_value(intccms)

In [ ]:
def skim_selection(puppimet, hlt):
    """Require a muon trigger and PuppiMET > 50 GeV."""
    selection = PackedSelection()
    selection.add("trigger", hlt.Mu50)
    selection.add("met_cut", puppimet.pt > 50)
    selection.add("skim", selection.all("trigger", "met_cut"))
    return selection

In [ ]:
class SkimProcessor(ProcessorABC):

    def __init__(self, skim_config, branches, output_dir):
        self.skim_config = skim_config
        self.output_dir = output_dir
        self.columns_to_keep, _ = build_column_list(branches)
        self.writer = RootWriter()

    @property
    def accumulator(self):
        return {"total_in": 0, "total_out": 0}

    def _build_output_path(self, events):
        file_hash = hashlib.md5(events.metadata["filename"].encode()).hexdigest()[:8]
        entry_start = events.metadata["entrystart"]
        entry_stop = events.metadata["entrystop"]
        dataset = events.metadata["dataset"]
        return f"{self.output_dir.rstrip('/')}/{dataset}/{file_hash}_{entry_start}_{entry_stop}"

    def process(self, events):
        output = self.accumulator
        dataset = events.metadata["dataset"]
        n_in = len(events)

        executor = SelectionExecutor(self.skim_config)
        mask = executor.execute(events)
        events = events[mask]
        n_out = len(events)

        if n_out > 0:
            output_columns = extract_columns(events, self.columns_to_keep)
            out_path = self._build_output_path(events)
            save_events(self.writer, output_columns, out_path, tree_name=TREE_NAME)

        output["total_in"] = n_in
        output["total_out"] = n_out
        print(f"  {dataset} [{events.metadata['entrystart']}:{events.metadata['entrystop']}]: {n_out}/{n_in} events passed")
        return output

    def postprocess(self, accumulator):
        return accumulator

In [ ]:
fileset = {DATASET_NAME: {"files": [INPUT_FILE], "treename": TREE_NAME}}

skim_config = SkimmingConfig(
    function=skim_selection,
    use=[("PuppiMET", None), ("HLT", None)],
)

processor = SkimProcessor(
    skim_config=skim_config,
    branches=BRANCHES,
    output_dir=OUTPUT_DIR,
)

t0 = time.perf_counter()

if AF == "iterative":
    runner = Runner(
        executor=IterativeExecutor(),
        schema=NanoAODSchema,
        chunksize=CHUNK_SIZE,
    )
    output = runner(fileset, treename=TREE_NAME, processor_instance=processor)
else:
    with acquire_client(AF, num_workers=NUM_WORKERS, close_after=AUTO_CLOSE_CLIENT) as (client, cluster):
        #stop = live_prints(client)
        runner = Runner(
            executor=DaskExecutor(client=client),
            schema=NanoAODSchema,
            chunksize=CHUNK_SIZE,
        )
        output = runner(fileset, treename=TREE_NAME, processor_instance=processor)
        #stop.set()

t1 = time.perf_counter()
total_in = output["total_in"]
total_out = output["total_out"]
print(f"\nDone in {t1-t0:.1f}s: {total_out:,}/{total_in:,} events kept ({100*total_out/total_in:.1f}%)")

In [ ]:
from intccms.skimming.io.readers import get_reader

reader = get_reader("ttree")
file_hash = hashlib.md5(INPUT_FILE.encode()).hexdigest()[:8]
test_file = f"{OUTPUT_DIR.rstrip('/')}/{DATASET_NAME}/{file_hash}_0_99231.root"
print(test_file)
events = reader.read(test_file, tree_name=TREE_NAME)
print(f"Read back {test_file.split('/')[-1]}: {len(events)} events, fields={events.fields}")

In [ ]:
if AF != "iterative" and not AUTO_CLOSE_CLIENT:
    client.close()